# Full mouse dataset quality summary: stop codons, AA identity, IgBLAST support/e-value

Запускать после готовности всех `*_igblast.tsv` для mouse dataset `ERP003950`.

Notebook считает **per-sample** и **overall по всем обработанным samples**:

- `pct_stop_codon`: доля reads со `stop_codon=True` (fallback: `*` в AA junction/CDR3, если колонки нет);
- `pct_bad_v_or_j_aa_identity`: доля reads, где V или J имеют AA identity `<85%`;
- `pct_bad_v_or_j_evalue`: доля reads, где IgBLAST `v_support` или `j_support` `>1`;
- `pct_bad_v_or_j_identity_or_evalue`: combined metric = AA identity `<85%` **OR** support/e-value `>1`;
- дополнительные полезные метрики: отдельно bad V/J, доступность AA identity/support полей, `productive=False`, `complete_vdj=False`, missing V/J calls.

Важно: в AIRR форматe поле называется `v_support/j_support`; для IgBLAST/Change-O это соответствует alignment E-value. Для других аннотаторов это было бы generic support, но здесь input — IgBLAST AIRR `outfmt 19`.


In [ ]:
from pathlib import Path
import csv, json, math
from datetime import datetime

AA_THRESHOLD = 85.0
EVALUE_THRESHOLD = 1.0
DATASET = "ERP003950"
BASE = Path("/data/user/epishkin/results") / DATASET
IG = BASE / "igblast"
Q = IG / "quality"
Q.mkdir(parents=True, exist_ok=True)

SAMPLES = ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]
RUN_INFO = {
    "dataset": DATASET,
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "input_dir": str(IG),
    "quality_dir": str(Q),
    "samples_expected": SAMPLES,
    "aa_identity_threshold_percent": AA_THRESHOLD,
    "evalue_support_threshold": EVALUE_THRESHOLD,
    "support_field_note": "В IgBLAST AIRR/Change-O выходе v_support/d_support/j_support — это alignment E-values. AIRR называет их generic support полями, потому что другие аннотаторы могут использовать p-value/likelihood/probability.",
}

def parse_bool(v):
    return str(v).strip().lower() in {"true", "t", "1", "yes", "y"}

def parse_float(v):
    if v is None:
        return None
    s = str(v).strip()
    if not s:
        return None
    try:
        x = float(s)
    except ValueError:
        return None
    if math.isnan(x):
        return None
    return x

def aa_identity_from_alignment(seq, germ):
    """Процент идентичности по выровненным AA query/germline строкам, позиции с gap/dot игнорируются."""
    if not seq or not germ:
        return None
    s, g = str(seq), str(germ)
    matches = aligned = 0
    for a, b in zip(s, g):
        if a in ".-" or b in ".-":
            continue
        aligned += 1
        if a == b:
            matches += 1
    return 100.0 * matches / aligned if aligned else None

def first_float(row, names):
    for name in names:
        x = parse_float(row.get(name))
        if x is not None:
            return x, name
    return None, None

def percent(n, d):
    return 100.0 * n / d if d else 0.0

summary_rows = []
bad_read_rows = []
input_status_rows = []

for sample in SAMPLES:
    p = IG / f"{sample}_igblast.tsv"
    status = {
        "sample": sample,
        "path": str(p),
        "exists": p.exists(),
        "bytes": p.stat().st_size if p.exists() else 0,
        "processed": False,
        "reason": "",
    }
    if not p.exists():
        status["reason"] = "missing"
        input_status_rows.append(status)
        print(f"SKIP {sample}: missing {p}")
        continue
    if p.stat().st_size < 1_000_000:
        status["reason"] = "too_small_lt_1MB_possible_partial"
        input_status_rows.append(status)
        print(f"SKIP {sample}: file too small ({p.stat().st_size} bytes), possible partial output")
        continue

    counters = {
        "n_records": 0,
        "n_stop_codon": 0,
        "n_productive_false": 0,
        "n_productive_available": 0,
        "n_complete_vdj_false": 0,
        "n_complete_vdj_available": 0,
        "n_v_aa_identity_available": 0,
        "n_j_aa_identity_available": 0,
        "n_bad_v_aa_identity": 0,
        "n_bad_j_aa_identity": 0,
        "n_bad_v_or_j_aa_identity": 0,
        "n_v_support_available": 0,
        "n_j_support_available": 0,
        "n_bad_v_evalue": 0,
        "n_bad_j_evalue": 0,
        "n_bad_v_or_j_evalue": 0,
        "n_bad_v_or_j_identity_or_evalue": 0,
        "n_bad_v_identity_or_evalue": 0,
        "n_bad_j_identity_or_evalue": 0,
        "n_missing_v_call": 0,
        "n_missing_j_call": 0,
    }

    with p.open(newline="") as f:
        reader = csv.DictReader(f, delimiter="\t")
        fieldnames = reader.fieldnames or []
        has_stop = "stop_codon" in fieldnames
        for row in reader:
            counters["n_records"] += 1
            n = counters["n_records"]
            sid = row.get("sequence_id", "")
            v_call = row.get("v_call", "")
            j_call = row.get("j_call", "")
            if not str(v_call).strip():
                counters["n_missing_v_call"] += 1
            if not str(j_call).strip():
                counters["n_missing_j_call"] += 1

            if has_stop:
                stop = parse_bool(row.get("stop_codon", ""))
            else:
                stop = "*" in (row.get("junction_aa") or row.get("cdr3_aa") or "")
            if stop:
                counters["n_stop_codon"] += 1

            if str(row.get("productive", "")).strip():
                counters["n_productive_available"] += 1
                if not parse_bool(row.get("productive")):
                    counters["n_productive_false"] += 1
            if str(row.get("complete_vdj", "")).strip():
                counters["n_complete_vdj_available"] += 1
                if not parse_bool(row.get("complete_vdj")):
                    counters["n_complete_vdj_false"] += 1

            v_aa_id, v_aa_source = first_float(row, ["v_identity_aa", "v_aa_identity", "v_sequence_identity_aa"])
            j_aa_id, j_aa_source = first_float(row, ["j_identity_aa", "j_aa_identity", "j_sequence_identity_aa"])
            if v_aa_id is None:
                v_aa_id = aa_identity_from_alignment(row.get("v_sequence_alignment_aa", ""), row.get("v_germline_alignment_aa", ""))
                v_aa_source = "alignment_aa" if v_aa_id is not None else ""
            if j_aa_id is None:
                j_aa_id = aa_identity_from_alignment(row.get("j_sequence_alignment_aa", ""), row.get("j_germline_alignment_aa", ""))
                j_aa_source = "alignment_aa" if j_aa_id is not None else ""

            if v_aa_id is not None:
                counters["n_v_aa_identity_available"] += 1
            if j_aa_id is not None:
                counters["n_j_aa_identity_available"] += 1
            bad_v_aa = v_aa_id is not None and v_aa_id < AA_THRESHOLD
            bad_j_aa = j_aa_id is not None and j_aa_id < AA_THRESHOLD
            if bad_v_aa:
                counters["n_bad_v_aa_identity"] += 1
            if bad_j_aa:
                counters["n_bad_j_aa_identity"] += 1
            bad_vj_aa = bad_v_aa or bad_j_aa
            if bad_vj_aa:
                counters["n_bad_v_or_j_aa_identity"] += 1

            v_support = parse_float(row.get("v_support"))
            j_support = parse_float(row.get("j_support"))
            if v_support is not None:
                counters["n_v_support_available"] += 1
            if j_support is not None:
                counters["n_j_support_available"] += 1
            bad_v_eval = v_support is not None and v_support > EVALUE_THRESHOLD
            bad_j_eval = j_support is not None and j_support > EVALUE_THRESHOLD
            if bad_v_eval:
                counters["n_bad_v_evalue"] += 1
            if bad_j_eval:
                counters["n_bad_j_evalue"] += 1
            bad_vj_eval = bad_v_eval or bad_j_eval
            if bad_vj_eval:
                counters["n_bad_v_or_j_evalue"] += 1

            bad_v_combined = bad_v_aa or bad_v_eval
            bad_j_combined = bad_j_aa or bad_j_eval
            bad_combined = bad_v_combined or bad_j_combined
            if bad_v_combined:
                counters["n_bad_v_identity_or_evalue"] += 1
            if bad_j_combined:
                counters["n_bad_j_identity_or_evalue"] += 1
            if bad_combined:
                counters["n_bad_v_or_j_identity_or_evalue"] += 1
                if len(bad_read_rows) < 200_000:
                    bad_read_rows.append({
                        "sample": sample,
                        "sequence_id": sid,
                        "v_call": v_call,
                        "j_call": j_call,
                        "productive": row.get("productive", ""),
                        "stop_codon": row.get("stop_codon", ""),
                        "v_aa_identity": f"{v_aa_id:.6f}" if v_aa_id is not None else "",
                        "j_aa_identity": f"{j_aa_id:.6f}" if j_aa_id is not None else "",
                        "v_aa_identity_source": v_aa_source,
                        "j_aa_identity_source": j_aa_source,
                        "v_support_evalue": f"{v_support:.6g}" if v_support is not None else "",
                        "j_support_evalue": f"{j_support:.6g}" if j_support is not None else "",
                        "bad_v_aa_identity": bad_v_aa,
                        "bad_j_aa_identity": bad_j_aa,
                        "bad_v_evalue": bad_v_eval,
                        "bad_j_evalue": bad_j_eval,
                        "bad_v_identity_or_evalue": bad_v_combined,
                        "bad_j_identity_or_evalue": bad_j_combined,
                    })

    n = counters["n_records"]
    row = {"sample": sample, **counters}
    # Проценты: основные метрики + полезные support-метрики.
    row.update({
        "pct_stop_codon": percent(counters["n_stop_codon"], n),
        "pct_productive_false": percent(counters["n_productive_false"], n),
        "pct_complete_vdj_false": percent(counters["n_complete_vdj_false"], n),
        "pct_bad_v_aa_identity": percent(counters["n_bad_v_aa_identity"], n),
        "pct_bad_j_aa_identity": percent(counters["n_bad_j_aa_identity"], n),
        "pct_bad_v_or_j_aa_identity": percent(counters["n_bad_v_or_j_aa_identity"], n),
        "pct_bad_v_evalue": percent(counters["n_bad_v_evalue"], n),
        "pct_bad_j_evalue": percent(counters["n_bad_j_evalue"], n),
        "pct_bad_v_or_j_evalue": percent(counters["n_bad_v_or_j_evalue"], n),
        "pct_bad_v_identity_or_evalue": percent(counters["n_bad_v_identity_or_evalue"], n),
        "pct_bad_j_identity_or_evalue": percent(counters["n_bad_j_identity_or_evalue"], n),
        "pct_bad_v_or_j_identity_or_evalue": percent(counters["n_bad_v_or_j_identity_or_evalue"], n),
        "pct_missing_v_call": percent(counters["n_missing_v_call"], n),
        "pct_missing_j_call": percent(counters["n_missing_j_call"], n),
        "aa_identity_threshold": AA_THRESHOLD,
        "evalue_support_threshold": EVALUE_THRESHOLD,
        "evalue_field_basis": "IgBLAST_AIRR_v_support_j_support",
    })
    summary_rows.append(row)
    status["processed"] = True
    status["reason"] = "ok"
    input_status_rows.append(status)
    print(
        f"{sample}: n={n} "
        f"stop={counters['n_stop_codon']} ({row['pct_stop_codon']:.3f}%) "
        f"badAA_VJ={counters['n_bad_v_or_j_aa_identity']} ({row['pct_bad_v_or_j_aa_identity']:.3f}%) "
        f"badEval_VJ={counters['n_bad_v_or_j_evalue']} ({row['pct_bad_v_or_j_evalue']:.3f}%) "
        f"badCombined={counters['n_bad_v_or_j_identity_or_evalue']} ({row['pct_bad_v_or_j_identity_or_evalue']:.3f}%)",
        flush=True,
    )

# Стабильный порядок колонок для per-sample summary.
summary_columns = [
    "sample", "n_records",
    "n_stop_codon", "pct_stop_codon",
    "n_productive_available", "n_productive_false", "pct_productive_false",
    "n_complete_vdj_available", "n_complete_vdj_false", "pct_complete_vdj_false",
    "n_v_aa_identity_available", "n_j_aa_identity_available",
    "n_bad_v_aa_identity", "pct_bad_v_aa_identity",
    "n_bad_j_aa_identity", "pct_bad_j_aa_identity",
    "n_bad_v_or_j_aa_identity", "pct_bad_v_or_j_aa_identity",
    "n_v_support_available", "n_j_support_available",
    "n_bad_v_evalue", "pct_bad_v_evalue",
    "n_bad_j_evalue", "pct_bad_j_evalue",
    "n_bad_v_or_j_evalue", "pct_bad_v_or_j_evalue",
    "n_bad_v_identity_or_evalue", "pct_bad_v_identity_or_evalue",
    "n_bad_j_identity_or_evalue", "pct_bad_j_identity_or_evalue",
    "n_bad_v_or_j_identity_or_evalue", "pct_bad_v_or_j_identity_or_evalue",
    "n_missing_v_call", "pct_missing_v_call",
    "n_missing_j_call", "pct_missing_j_call",
    "aa_identity_threshold", "evalue_support_threshold", "evalue_field_basis",
]

def write_tsv(path, rows, fieldnames=None):
    path = Path(path)
    if fieldnames is None:
        fieldnames = list(rows[0].keys()) if rows else []
    with path.open("w", newline="") as f:
        if not fieldnames:
            return
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t", extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

write_tsv(Q / "mouse_quality_summary_by_sample.tsv", summary_rows, summary_columns)
# Таблицы по отдельным метрикам под старыми именами файлов, но с расширенными полями (для обратной совместимости).
write_tsv(Q / "mouse_stop_codon_summary.tsv", summary_rows, ["sample", "n_records", "n_stop_codon", "pct_stop_codon"])
write_tsv(Q / "mouse_vj_quality_summary.tsv", summary_rows, [
    "sample", "n_records",
    "n_bad_v_or_j_aa_identity", "pct_bad_v_or_j_aa_identity",
    "n_bad_v_aa_identity", "n_bad_j_aa_identity",
    "n_v_aa_identity_available", "n_j_aa_identity_available",
    "n_bad_v_or_j_evalue", "pct_bad_v_or_j_evalue",
    "n_bad_v_evalue", "n_bad_j_evalue",
    "n_v_support_available", "n_j_support_available",
    "n_bad_v_or_j_identity_or_evalue", "pct_bad_v_or_j_identity_or_evalue",
    "aa_identity_threshold", "evalue_support_threshold", "evalue_field_basis",
])
write_tsv(Q / "mouse_bad_vj_reads.tsv", bad_read_rows)
write_tsv(Q / "mouse_input_status.tsv", input_status_rows)

# Overall-сводка по всем обработанным samples.
sum_fields = [c for c in summary_columns if c.startswith("n_")]
overall = {"scope": "all_processed_samples", "samples_processed": [r["sample"] for r in summary_rows]}
for field in sum_fields:
    overall[field] = sum(int(r.get(field, 0)) for r in summary_rows)
n = overall.get("n_records", 0)
overall.update({
    "pct_stop_codon": percent(overall["n_stop_codon"], n),
    "pct_productive_false": percent(overall["n_productive_false"], n),
    "pct_complete_vdj_false": percent(overall["n_complete_vdj_false"], n),
    "pct_bad_v_aa_identity": percent(overall["n_bad_v_aa_identity"], n),
    "pct_bad_j_aa_identity": percent(overall["n_bad_j_aa_identity"], n),
    "pct_bad_v_or_j_aa_identity": percent(overall["n_bad_v_or_j_aa_identity"], n),
    "pct_bad_v_evalue": percent(overall["n_bad_v_evalue"], n),
    "pct_bad_j_evalue": percent(overall["n_bad_j_evalue"], n),
    "pct_bad_v_or_j_evalue": percent(overall["n_bad_v_or_j_evalue"], n),
    "pct_bad_v_identity_or_evalue": percent(overall["n_bad_v_identity_or_evalue"], n),
    "pct_bad_j_identity_or_evalue": percent(overall["n_bad_j_identity_or_evalue"], n),
    "pct_bad_v_or_j_identity_or_evalue": percent(overall["n_bad_v_or_j_identity_or_evalue"], n),
    "pct_missing_v_call": percent(overall["n_missing_v_call"], n),
    "pct_missing_j_call": percent(overall["n_missing_j_call"], n),
    "aa_identity_threshold": AA_THRESHOLD,
    "evalue_support_threshold": EVALUE_THRESHOLD,
    "evalue_field_basis": "IgBLAST_AIRR_v_support_j_support",
})

overall_columns = ["scope", "samples_processed"] + [c for c in summary_columns if c != "sample"]
write_tsv(Q / "mouse_quality_summary_overall.tsv", [overall], overall_columns)

payload = {"run_info": RUN_INFO, "input_status": input_status_rows, "per_sample": summary_rows, "overall": overall}
(Q / "mouse_quality_summary.json").write_text(json.dumps(payload, indent=2, ensure_ascii=False))
# JSON-файлы под старыми именами (для обратной совместимости).
(Q / "mouse_stop_codon_summary.json").write_text(json.dumps({"run_info": RUN_INFO, "per_sample": summary_rows, "overall": overall}, indent=2, ensure_ascii=False))
(Q / "mouse_vj_quality_summary.json").write_text(json.dumps({"run_info": RUN_INFO, "per_sample": summary_rows, "overall": overall}, indent=2, ensure_ascii=False))

print("\n=== OVERALL ===")
print(json.dumps({
    "n_records": overall["n_records"],
    "pct_stop_codon": overall["pct_stop_codon"],
    "pct_bad_v_or_j_aa_identity": overall["pct_bad_v_or_j_aa_identity"],
    "pct_bad_v_or_j_evalue": overall["pct_bad_v_or_j_evalue"],
    "pct_bad_v_or_j_identity_or_evalue": overall["pct_bad_v_or_j_identity_or_evalue"],
    "pct_productive_false": overall["pct_productive_false"],
    "aa_identity_threshold": AA_THRESHOLD,
    "evalue_support_threshold": EVALUE_THRESHOLD,
}, indent=2))
print("\nWritten files:")
for name in [
    "mouse_quality_summary_by_sample.tsv",
    "mouse_quality_summary_overall.tsv",
    "mouse_quality_summary.json",
    "mouse_stop_codon_summary.tsv",
    "mouse_vj_quality_summary.tsv",
    "mouse_bad_vj_reads.tsv",
    "mouse_input_status.tsv",
]:
    p = Q / name
    print(p, p.stat().st_size if p.exists() else "MISSING")


In [ ]:
from pathlib import Path
Q = Path("/data/user/epishkin/results/ERP003950/igblast/quality")
for name in ["mouse_quality_summary_by_sample.tsv", "mouse_quality_summary_overall.tsv", "mouse_stop_codon_summary.tsv", "mouse_vj_quality_summary.tsv", "mouse_input_status.tsv"]:
    f = Q / name
    print("\n==", name, "==")
    if f.exists():
        text = f.read_text().splitlines()
        print("\n".join(text[:12]))
    else:
        print("MISSING")
